# Traffix.id — LSTM Inference Pipeline

## 1. Install Dependencies

In [1]:
!pip install tensorflow joblib pandas numpy -q
!pip install tf-keras -q
print("Dependencies installed.")

Dependencies installed.


## 2. Imports

In [2]:
import os, json, math, warnings
import numpy as np
import pandas as pd
import joblib
import tensorflow as tf
from datetime import datetime, timedelta

warnings.filterwarnings('ignore')

print("Imports OK.")
print(f"TensorFlow version: {tf.__version__}")

Imports OK.
TensorFlow version: 2.20.0


## 3. Upload Artifacts


In [3]:
from google.colab import files as colab_files

ARTIFACTS_DIR = '/content/artifacts'
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

print("Upload artifact:")
print("  1. best_lstm_15m.keras")
print("  2. best_lstm_2h.keras")
print("  3. best_lstm_4h.keras")
print("  4. feat_scaler.joblib")
print("  5. target_scalers.joblib")
print("  6. best_config.json")
print("  7. feature_columns.json")
print("  8. vehicle_count.csv  (output dari notebook YOLO inference)")
print()

uploaded = colab_files.upload()
for fname, data in uploaded.items():
    dest = os.path.join(ARTIFACTS_DIR, fname)
    with open(dest, 'wb') as f:
        f.write(data)
    print(f"Saved: {dest}")

Upload artifact:
  1. best_lstm_15m.keras
  2. best_lstm_2h.keras
  3. best_lstm_4h.keras
  4. feat_scaler.joblib
  5. target_scalers.joblib
  6. best_config.json
  7. feature_columns.json
  8. vehicle_count.csv  (output dari notebook YOLO inference)



Saving best_config.json to best_config.json
Saving best_lstm_2h.keras to best_lstm_2h.keras
Saving best_lstm_4h.keras to best_lstm_4h.keras
Saving best_lstm_15m.keras to best_lstm_15m.keras
Saving feat_scaler.joblib to feat_scaler.joblib
Saving feature_columns.json to feature_columns.json
Saving target_scalers.joblib to target_scalers.joblib
Saving vehicle_count.csv to vehicle_count.csv
Saved: /content/artifacts/best_config.json
Saved: /content/artifacts/best_lstm_2h.keras
Saved: /content/artifacts/best_lstm_4h.keras
Saved: /content/artifacts/best_lstm_15m.keras
Saved: /content/artifacts/feat_scaler.joblib
Saved: /content/artifacts/feature_columns.json
Saved: /content/artifacts/target_scalers.joblib
Saved: /content/artifacts/vehicle_count.csv


## 4. Runtime Configuration & Output Paths

In [4]:
OUTPUT_DIR = '/content/outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Daftar artifact wajib
REQUIRED_FILES = [
    'best_lstm_15m.keras', 'best_lstm_2h.keras', 'best_lstm_4h.keras',
    'feat_scaler.joblib', 'target_scalers.joblib',
    'best_config.json', 'feature_columns.json',
    'vehicle_count.csv',
]

missing = [f for f in REQUIRED_FILES if not os.path.exists(os.path.join(ARTIFACTS_DIR, f))]
if missing:
    print(f"[WARNING] File berikut belum diupload: {missing}")
else:
    print("[OK] Semua artifact yang diperlukan sudah tersedia.")

print(f"Output dir: {OUTPUT_DIR}")

[OK] Semua artifact yang diperlukan sudah tersedia.
Output dir: /content/outputs


## 5. Load LSTM Artifacts (Model 15m / 2h / 4h + Scalers)

In [5]:
with open(os.path.join(ARTIFACTS_DIR, 'best_config.json')) as f:
    config = json.load(f)

SEQ_LEN = config['seq_len']
print(f"Config: seq_len={SEQ_LEN}")

with open(os.path.join(ARTIFACTS_DIR, 'feature_columns.json')) as f:
    FEATURE_COLUMNS = json.load(f)
NUM_FEATURES = len(FEATURE_COLUMNS)
print(f"Features: {NUM_FEATURES} columns")

feat_scaler    = joblib.load(os.path.join(ARTIFACTS_DIR, 'feat_scaler.joblib'))
target_scalers = joblib.load(os.path.join(ARTIFACTS_DIR, 'target_scalers.joblib'))
print(f"Scalers loaded. target_scalers keys: {list(target_scalers.keys())}")

# MAPE per horizon.
HORIZON_MAPE = {
    '15m': config.get('test_mape_15m'),
    '2h':  config.get('test_mape_2h', 27.035),
    '4h':  config.get('test_mape_4h', 32.492),
}
print(f"MAPE per horizon: {HORIZON_MAPE}")

# Load 3 model LSTM (15m, 2h, 4h)
LSTM_MODELS = {}
for h, fname in [('15m', 'best_lstm_15m.keras'),
                  ('2h',  'best_lstm_2h.keras'),
                  ('4h',  'best_lstm_4h.keras')]:
    LSTM_MODELS[h] = tf.keras.models.load_model(
        os.path.join(ARTIFACTS_DIR, fname), compile=False)
    print(f"[OK] LSTM model '{h}' loaded from {fname} | input_shape={LSTM_MODELS[h].input_shape}")

print()
LSTM_MODELS['15m'].summary()

Config: seq_len=15
Features: 41 columns
Scalers loaded. target_scalers keys: ['15m', '2h', '4h']
MAPE per horizon: {'15m': 21.8059, '2h': 27.035, '4h': 32.492}
[OK] LSTM model '15m' loaded from best_lstm_15m.keras | input_shape=(None, 15, 41)
[OK] LSTM model '2h' loaded from best_lstm_2h.keras | input_shape=(None, 15, 41)
[OK] LSTM model '4h' loaded from best_lstm_4h.keras | input_shape=(None, 15, 41)



Model: "lstm_light"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input (InputLayer)              │ (None, 15, 41)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm1 (LSTM)                    │ (None, 15, 64)         │        27,136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ drop1 (Dropout)                 │ (None, 15, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm2 (LSTM)                    │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense1 (Dense)                  │ (None, 64)             │         2,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ drop2 (Dropout)                 │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 41,729 (163.00 KB)

 Trainable params: 41,729 (163.00 KB)

 Non-trainable params: 0 (0.00 B)

## 6. LSTM Sequence Builder (Training-Aligned Synthetic Sequence)

In [6]:
def row_id_to_seed(row_id):
    return abs(hash(row_id)) % (2**31)

def build_training_aligned_sequence(obs, seq_len, rng):
    vc   = float(obs.get('vehicle_count_1min', 30.0))
    vol  = vc * 60.0
    spd  = float(obs.get('avg_speed_kmh', 50.0))
    q    = float(obs.get('queue_length_veh', 0.0))
    den  = float(obs.get('density_percent', 20.0))

    # Reconstruct base datetime (step i=seq_len-1 is "now")
    try:
        obs_dt = datetime(
            int(obs.get('year', 2024)), int(obs.get('month', 5)),
            int(obs.get('day', 1)), int(obs.get('hour', 8)), int(obs.get('minute', 0))
        )
    except Exception:
        obs_dt = datetime(2024, 5, 1, int(obs.get('hour', 8)), int(obs.get('minute', 0)))

    # Step 1: Generate seq_len-step volume_veh_per_hour history
    # BUG-A FIX: when vol=0, keep history at 0 (no noise inflation)
    noise_pct = 0.04
    vols = []
    if vol <= 0:
        vols = [0.0] * seq_len
    else:
        for i in range(seq_len):
            ramp  = 0.70 + 0.30 * (i / max(seq_len - 1, 1))
            base  = vol * ramp
            noise = rng.normal(0, max(abs(base) * noise_pct, 5.0))
            vols.append(max(0.0, base + noise))
        vols[-1] = vol    # anchor last step to actual observation

    vol_s = np.array(vols)

    # BUG-B FIX: safe denominator prevents explosion when vol=0 but q or den > 0
    vol_denom = max(vol, 1.0)

    SPEED_BASE = max(spd, 10.0)
    VOL_BASE   = max(vol, 60.0)
    spds = [max(5.0, SPEED_BASE * (1.0 - 0.4 * (vol_s[i] - vol_s[-1]) / VOL_BASE))
            for i in range(seq_len)]
    spds[-1] = spd

    qs   = [q * (vol_s[i] / vol_denom) for i in range(seq_len)]
    qs[-1] = q
    dens = [min(100.0, den * (vol_s[i] / vol_denom)) for i in range(seq_len)]
    dens[-1] = den

    # Step 2: Build per-timestep feature rows
    rows = []
    for i in range(seq_len):
        vc_i  = vol_s[i] / 60.0
        vol_i = vol_s[i]
        spd_i = spds[i]
        q_i   = qs[i]
        den_i = dens[i]

        wait_i  = q_i * 0.5
        green_i = float(obs.get('green_seconds', 25.6))

        dv = vol_s[i] - vol_s[i-1] if i > 0 else 0.0

        def lv(lag): return float(vol_s[max(i - lag, 0)])
        lag1  = lv(1);  lag5  = lv(5)
        lag15 = lv(15); lag30 = lv(30); lag60 = lv(60)

        lag_spd15 = float(spds[max(i-15, 0)])
        lag_q15   = float(qs[max(i-15, 0)])

        w15 = vol_s[max(0, i-14):i+1]
        w60 = vol_s[max(0, i-59):i+1]

        # BUG-C FIX: reconstruct per-step timestamp
        # step i=seq_len-1 is "now"; earlier steps go back in time
        minutes_ago = (seq_len - 1) - i
        step_dt     = obs_dt - timedelta(minutes=minutes_ago)
        h_i         = step_dt.hour
        m_i         = step_dt.minute
        sin_i       = math.sin(2 * math.pi * h_i / 24)
        cos_i       = math.cos(2 * math.pi * h_i / 24)

        row = {
            'vehicle_count_1min':   vc_i,
            'volume_veh_per_hour':  vol_i,
            'avg_speed_kmh':        spd_i,
            'queue_length_veh':     q_i,
            'wait_time_min':        wait_i,
            'green_seconds':        green_i,
            'density_percent':      den_i,
            'weather_temp_c':       float(obs.get('weather_temp_c', 30.0)),
            'accident_count':       float(obs.get('accident_count', 0)),
            'roadwork_flag':        float(obs.get('roadwork_flag', 0)),
            'event_flag':           float(obs.get('event_flag', 0)),
            'hour':                 float(h_i),
            'minute':               float(m_i),
            'day':                  float(obs.get('day', 1)),
            'day_of_week':          float(obs.get('day_of_week', 1)),
            'month':                float(obs.get('month', 6)),
            'is_holiday':           float(obs.get('is_holiday', 0)),
            'is_weekend':           float(obs.get('is_weekend', 0)),
            'hour_sin':             sin_i,
            'hour_cos':             cos_i,
            'delta_volume':         dv,
            'lag_1':               lag1,
            'lag_5':               lag5,
            'lag_15':              lag15,
            'lag_30':              lag30,
            'lag_60':              lag60,
            'lag_speed_15':        lag_spd15,
            'lag_queue_15':        lag_q15,
            'roll_mean_15':        float(w15.mean()),
            'roll_std_15':         float(w15.std()) if len(w15) > 1 else 0.0,
            'roll_min_15':         float(w15.min()),
            'roll_max_15':         float(w15.max()),
            'roll_median_15':      float(np.median(w15)),
            'roll_mean_60':        float(w60.mean()),
            'roll_std_60':         float(w60.std()) if len(w60) > 1 else 0.0,
            'roll_min_60':         float(w60.min()),
            'roll_max_60':         float(w60.max()),
            'weather_condition_Clear':  float(obs.get('weather_condition_Clear', 1)),
            'weather_condition_Cloudy': float(obs.get('weather_condition_Cloudy', 0)),
            'weather_condition_Hot':    float(obs.get('weather_condition_Hot', 0)),
            'weather_condition_Rain':   float(obs.get('weather_condition_Rain', 0)),
        }
        rows.append(row)

    # Step 3: Assemble array in exact FEATURE_COLUMNS order
    arr = np.array([[r[col] for col in FEATURE_COLUMNS] for r in rows], dtype=np.float64)
    return arr    # shape: (seq_len, n_features)

print("build_training_aligned_sequence() ready.")

build_training_aligned_sequence() ready.


## 7. Multi-Horizon Prediction Function (15m / 2h / 4h)

In [7]:
def predict_from_obs_multi(obs, row_id, verbose=True):
    rng = np.random.default_rng(row_id_to_seed(row_id))
    seq = build_training_aligned_sequence(obs, SEQ_LEN, rng)

    expected_feat_count = len(FEATURE_COLUMNS)
    if seq.shape != (SEQ_LEN, expected_feat_count):
        raise ValueError(
            f"[ERROR] raw sequence shape {seq.shape} != "
            f"({SEQ_LEN}, {expected_feat_count}). Check FEATURE_COLUMNS."
        )

    if not np.isfinite(seq).all():
        seq = np.nan_to_num(seq, nan=0.0, posinf=0.0, neginf=0.0)

    scaled = feat_scaler.transform(seq)
    X = scaled.reshape(1, SEQ_LEN, NUM_FEATURES).astype(np.float32)

    results = {}
    for h, model in LSTM_MODELS.items():
        pred_scaled = model.predict(X, verbose=0)
        scaler_h = target_scalers[h]
        inv = float(scaler_h.inverse_transform(pred_scaled.reshape(-1, 1)).flatten()[0])
        results[h] = max(0.0, round(inv, 2))

    if verbose:
        print(f"    -> 15m={results['15m']:.1f} | 2h={results['2h']:.1f} | 4h={results['4h']:.1f} veh/hour")

    return results

print("predict_from_obs_multi() ready.")

predict_from_obs_multi() ready.


## 8. Sensitivity Test (Sanity Check)

In [8]:
print("=== LSTM SENSITIVITY TEST (15m / 2h / 4h) ===")
print("Expected: predictions vary proportionally with vehicle_count_1min")
print()

_base_hour = 8
_base_obs = {
    'avg_speed_kmh':    50.0,
    'queue_length_veh': 2.0,
    'density_percent':  20.0,
    'weather_temp_c':   30.0,
    'hour':             _base_hour,
    'minute':           0,
    'day':              18,
    'day_of_week':      0,
    'month':            5,
    'is_holiday':       0,
    'is_weekend':       0,
    'hour_sin':         float(np.sin(2*np.pi*_base_hour/24)),
    'hour_cos':         float(np.cos(2*np.pi*_base_hour/24)),
    'weather_condition_Clear': 1, 'weather_condition_Cloudy': 0,
    'weather_condition_Hot': 0,   'weather_condition_Rain': 0,
    'accident_count': 0, 'roadwork_flag': 0, 'event_flag': 0,
    'green_seconds': 25.6, 'wait_time_min': 1.0,
}

_scenarios = [
    ('A',   0, "empty road"),
    ('B',  20, "light traffic"),
    ('C',  40, "moderate traffic"),
    ('D',  80, "heavy traffic"),
    ('E', 120, "very heavy traffic"),
]

_res_15m, _res_2h, _res_4h, _vc_vals = [], [], [], []

print(f"  {'Scenario':<22} {'vc_1min':>8}  {'15m':>9}  {'2h':>9}  {'4h':>9}")
print(f"  {'-'*22} {'-'*8}  {'-'*9}  {'-'*9}  {'-'*9}")
for label, vc_val, desc in _scenarios:
    obs = dict(_base_obs)
    obs['vehicle_count_1min']  = float(vc_val)
    obs['volume_veh_per_hour'] = float(vc_val * 60)
    obs['queue_length_veh']    = max(0.0, vc_val * 0.15)
    obs['density_percent']     = min(100.0, vc_val / 1.7)
    obs['avg_speed_kmh']       = max(10.0, 65.0 - vc_val * 0.35)
    obs['wait_time_min']       = max(0.0, vc_val * 0.07)

    res = predict_from_obs_multi(obs, f"sensitivity_{label}", verbose=False)
    _res_15m.append(res['15m']); _res_2h.append(res['2h']); _res_4h.append(res['4h'])
    _vc_vals.append(vc_val)
    print(f"  {label}: {desc:<18} {vc_val:>8.0f}  {res['15m']:>9.1f}  {res['2h']:>9.1f}  {res['4h']:>9.1f}")

print()
for label_h, vals in [('15m', _res_15m), ('2h', _res_2h), ('4h', _res_4h)]:
    spread = max(vals) - min(vals)
    mean_v = sum(vals) / len(vals)
    rel = spread / max(abs(mean_v), 1.0)
    if rel < 0.05:
        print(f"[WARNING] Horizon {label_h}: predictions nearly constant "
              f"(spread={round(spread,1)}, {round(rel*100,1)}%).")
    else:
        print(f"[OK] Horizon {label_h}: spread={round(spread,1)} veh/hour "
              f"({round(rel*100,1)}% relative). Model responds to traffic load.")

print("\nLSTM helpers ready.")

=== LSTM SENSITIVITY TEST (15m / 2h / 4h) ===
Expected: predictions vary proportionally with vehicle_count_1min

  Scenario                vc_1min        15m         2h         4h
  ---------------------- --------  ---------  ---------  ---------
  A: empty road                0      350.0      514.1      752.1
  B: light traffic            20      967.4     1859.0     1630.6
  C: moderate traffic         40     2817.3     3255.7     2168.2
  D: heavy traffic            80     3540.9     2147.3     3284.3
  E: very heavy traffic      120     3784.3     1758.8     4024.4

[OK] Horizon 15m: spread=3434.3 veh/hour (149.8% relative). Model responds to traffic load.
[OK] Horizon 2h: spread=2741.6 veh/hour (143.8% relative). Model responds to traffic load.
[OK] Horizon 4h: spread=3272.3 veh/hour (138.0% relative). Model responds to traffic load.

LSTM helpers ready.


## 9. Load Vehicle Count Data (Output YOLO Inference)

In [9]:
VC_PATH = os.path.join(ARTIFACTS_DIR, 'vehicle_count.csv')
vc_df = pd.read_csv(VC_PATH, encoding='utf-8-sig')

print(f"Loaded  : {VC_PATH}")
print(f"Rows    : {len(vc_df)}")
print(f"Columns : {list(vc_df.columns)}")
vc_df.head()

Loaded  : /content/artifacts/vehicle_count.csv
Rows    : 100
Columns : ['camera_id', 'source_video', 'date', 'time', 'period', 'latitude', 'longitude', 'weather', 'weather_temp_c', 'vehicle_count', 'vehicle_count_1min', 'volume_veh_per_hour', 'avg_speed_kmh', 'density_percent', 'queue_length_veh', 'congestion_level', 'detected_video_path']


,camera_id,source_video,date,time,period,latitude,longitude,weather,weather_temp_c,vehicle_count,vehicle_count_1min,volume_veh_per_hour,avg_speed_kmh,density_percent,queue_length_veh,congestion_level,detected_video_path
0,CAM_002,pagi_cctv_001_20260518_070935.mp4,5/18/2026,6:09:33,Morning Rush,-6.213143,106.683766,Cloudy,25.6,23,72,4320,53.62,19.80,0,Low,/content/outputs/detected_videos/detected_pagi...
1,CAM_001,pagi_cctv_002_20260518_070954.mp4,5/18/2026,6:15:26,Morning Rush,-6.191151,106.744189,Cloudy,26.8,67,71,4260,40.17,51.03,1,Medium,/content/outputs/detected_videos/detected_pagi...
2,CAM_001,pagi_cctv_003_20260614_080117.mp4,6/14/2026,8:01:17,Morning Rush,-6.191151,106.744189,Cloudy,27.8,9,34,2040,46.51,7.59,0,Low,/content/outputs/detected_videos/detected_pagi...
3,CAM_011,pagi_cctv_004_20260518_071356.mp4,5/18/2026,6:18:01,Morning Rush,-6.191635,106.730124,Cloudy,25.6,63,73,4380,35.38,51.35,1,Medium,/content/outputs/detected_videos/detected_pagi...
4,CAM_007,pagi_cctv_005_20260614_080139.mp4,6/14/2026,8:01:39,Morning Rush,-6.202452,106.705216,Cloudy,27.8,54,85,5100,78.82,45.45,0,Low,/content/outputs/detected_videos/detected_pagi...


## 10. Green Signal Duration Estimator (Adaptive)

In [10]:
GREEN_BASE = 45.0   # detik — durasi default pada kondisi lalu lintas normal
GREEN_MIN  = 20.0   # detik — batas bawah durasi hijau
GREEN_MAX  = 90.0   # detik — batas atas durasi hijau

# Baseline kondisi "normal" yang menghasilkan GREEN_BASE
_DENSITY_BASELINE = 20.0   # %
_SPEED_BASELINE   = 50.0   # km/h

def compute_green_seconds(density_percent, queue_length_veh, avg_speed_kmh):
    density_adj = (density_percent - _DENSITY_BASELINE) * 0.30
    queue_adj   = queue_length_veh * 1.5
    speed_adj   = (_SPEED_BASELINE - avg_speed_kmh) * 0.20

    green = GREEN_BASE + density_adj + queue_adj + speed_adj
    return float(round(min(GREEN_MAX, max(GREEN_MIN, green)), 1))

# Sanity check: kondisi normal -> menghasilkan GREEN_BASE (45s)
_check = compute_green_seconds(_DENSITY_BASELINE, 0, _SPEED_BASELINE)
print(f"Sanity check (density=20%, queue=0, speed=50 km/h) -> green_seconds = {_check}s "
      f"(expected {GREEN_BASE}s)")

Sanity check (density=20%, queue=0, speed=50 km/h) -> green_seconds = 45.0s (expected 45.0s)


## 11. Build Per-Row Feature Observation (`obs`)

In [11]:
WEATHER_CONDITIONS = ['Clear', 'Cloudy', 'Hot', 'Rain']

def build_obs_from_row(row):
    # Parse date & time
    try:
        # Clean the time string by removing ' AM' or ' PM' if present
        time_str = str(row['time'])
        if ' AM' in time_str:
            time_str = time_str.replace(' AM', '')
        elif ' PM' in time_str:
            time_str = time_str.replace(' PM', '')

        dt_obj = datetime.strptime(f"{row['date']} {time_str}", '%m/%d/%Y %H:%M:%S')
    except Exception:
        dt_obj = datetime.now()

    hour, minute = dt_obj.hour, dt_obj.minute
    day, month   = dt_obj.day, dt_obj.month
    day_of_week  = dt_obj.weekday()
    is_weekend   = 1 if day_of_week >= 5 else 0

    # Weather one-hot
    weather_str = str(row.get('weather', 'Clear'))
    weather_onehot = {f'weather_condition_{c}': (1.0 if weather_str == c else 0.0)
                      for c in WEATHER_CONDITIONS}

    # Core traffic features dari hasil YOLO
    queue_length_veh = float(row.get('queue_length_veh', 0.0))
    avg_speed_kmh    = float(row.get('avg_speed_kmh', 0.0))
    density_percent  = float(row.get('density_percent', 0.0))

    # Derived features
    wait_time_min = round(queue_length_veh * 0.4, 2)
    green_seconds = compute_green_seconds(density_percent, queue_length_veh, avg_speed_kmh)

    obs = {
        'vehicle_count_1min':  float(row.get('vehicle_count_1min', 0.0)),
        'volume_veh_per_hour': float(row.get('volume_veh_per_hour', 0.0)),
        'avg_speed_kmh':       avg_speed_kmh,
        'queue_length_veh':    queue_length_veh,
        'wait_time_min':       wait_time_min,
        'green_seconds':       green_seconds,
        'density_percent':     density_percent,
        'weather_temp_c':      float(row.get('weather_temp_c', 30.0)),
        'accident_count':      float(row.get('accident_count', 0)),
        'roadwork_flag':       float(row.get('roadwork_flag', 0)),
        'event_flag':          float(row.get('event_flag', 0)),
        'is_holiday':          float(row.get('is_holiday', 0)),
        'hour':         hour,
        'minute':       minute,
        'day':          day,
        'day_of_week':  day_of_week,
        'month':        month,
        'is_weekend':   is_weekend,
        'hour_sin':     float(np.sin(2*np.pi*hour/24)),
        'hour_cos':     float(np.cos(2*np.pi*hour/24)),
    }
    obs.update(weather_onehot)
    return obs

print("build_obs_from_row() ready.")

build_obs_from_row() ready.


## 12. Run LSTM Predictions per Baris (15m / 2h / 4h)

In [12]:
print("=== LSTM PREDICTION (15m / 2h / 4h) PER ROW ===")

pred_15m, pred_2h, pred_4h = [], [], []
green_seconds_list = []
feature_rows = []

for idx, row in vc_df.iterrows():
    row_id = row.get('source_video', f'row_{idx}')
    obs = build_obs_from_row(row)

    print(f"\n  [{idx+1}/{len(vc_df)}] {row_id}")
    print(f"    vehicle_count_1min  = {obs['vehicle_count_1min']:.2f}")
    print(f"    volume_veh_per_hour = {obs['volume_veh_per_hour']:.2f}")
    print(f"    avg_speed_kmh       = {obs['avg_speed_kmh']:.2f}")
    print(f"    density_percent     = {obs['density_percent']:.2f}")
    print(f"    green_seconds       = {obs['green_seconds']:.1f}")

    try:
        res = predict_from_obs_multi(obs, row_id)
    except Exception as e:
        print(f"    [ERROR] {e}")
        res = {'15m': 0.0, '2h': 0.0, '4h': 0.0}

    pred_15m.append(res['15m'])
    pred_2h.append(res['2h'])
    pred_4h.append(res['4h'])
    green_seconds_list.append(obs['green_seconds'])

    # Simpan feature row (obs) + identitas baris untuk traffic_features CSV
    feat_row = {
        'camera_id':    row.get('camera_id', f'CAM_{idx+1:03d}'),
        'source_video': row.get('source_video', ''),
        'date':         row.get('date'),
        'time':         row.get('time'),
        'period':       row.get('period'),
    }
    feat_row.update(obs)
    feature_rows.append(feat_row)

vc_df['predicted_volume_15m'] = pred_15m
vc_df['predicted_volume_2h']  = pred_2h
vc_df['predicted_volume_4h']  = pred_4h
vc_df['green_seconds']        = green_seconds_list

print("\n[OK] Prediksi selesai untuk semua baris.")

=== LSTM PREDICTION (15m / 2h / 4h) PER ROW ===

  [1/100] pagi_cctv_001_20260518_070935.mp4
    vehicle_count_1min  = 72.00
    volume_veh_per_hour = 4320.00
    avg_speed_kmh       = 53.62
    density_percent     = 19.80
    green_seconds       = 44.2
    -> 15m=4506.8 | 2h=1585.8 | 4h=4978.1 veh/hour

  [2/100] pagi_cctv_002_20260518_070954.mp4
    vehicle_count_1min  = 71.00
    volume_veh_per_hour = 4260.00
    avg_speed_kmh       = 40.17
    density_percent     = 51.03
    green_seconds       = 57.8
    -> 15m=3870.7 | 2h=1628.0 | 4h=4986.5 veh/hour

  [3/100] pagi_cctv_003_20260614_080117.mp4
    vehicle_count_1min  = 34.00
    volume_veh_per_hour = 2040.00
    avg_speed_kmh       = 46.51
    density_percent     = 7.59
    green_seconds       = 42.0
    -> 15m=1750.7 | 2h=1215.8 | 4h=4722.3 veh/hour

  [4/100] pagi_cctv_004_20260518_071356.mp4
    vehicle_count_1min  = 73.00
    volume_veh_per_hour = 4380.00
    avg_speed_kmh       = 35.38
    density_percent     = 51.35
    gre

## 13. Save Traffic Feature CSV

In [13]:
df_feat = pd.DataFrame(feature_rows)

feat_csv_path = os.path.join(OUTPUT_DIR, 'traffic_features_lstm.csv')
df_feat.to_csv(feat_csv_path, index=False)
print(f"Saved: {feat_csv_path}")
print(f"Shape: {df_feat.shape}")
df_feat.head()

Saved: /content/outputs/traffic_features_lstm.csv
Shape: (100, 29)


,camera_id,source_video,date,time,period,vehicle_count_1min,volume_veh_per_hour,avg_speed_kmh,queue_length_veh,wait_time_min,...,day,day_of_week,month,is_weekend,hour_sin,hour_cos,weather_condition_Clear,weather_condition_Cloudy,weather_condition_Hot,weather_condition_Rain
0,CAM_002,pagi_cctv_001_20260518_070935.mp4,5/18/2026,6:09:33,Morning Rush,72.0,4320.0,53.62,0.0,0.0,...,15,0,6,0,0.866025,0.5,0.0,1.0,0.0,0.0
1,CAM_001,pagi_cctv_002_20260518_070954.mp4,5/18/2026,6:15:26,Morning Rush,71.0,4260.0,40.17,1.0,0.4,...,15,0,6,0,0.866025,0.5,0.0,1.0,0.0,0.0
2,CAM_001,pagi_cctv_003_20260614_080117.mp4,6/14/2026,8:01:17,Morning Rush,34.0,2040.0,46.51,0.0,0.0,...,15,0,6,0,0.866025,0.5,0.0,1.0,0.0,0.0
3,CAM_011,pagi_cctv_004_20260518_071356.mp4,5/18/2026,6:18:01,Morning Rush,73.0,4380.0,35.38,1.0,0.4,...,15,0,6,0,0.866025,0.5,0.0,1.0,0.0,0.0
4,CAM_007,pagi_cctv_005_20260614_080139.mp4,6/14/2026,8:01:39,Morning Rush,85.0,5100.0,78.82,0.0,0.0,...,15,0,6,0,0.866025,0.5,0.0,1.0,0.0,0.0


## 14. Congestion Level, AI Insight & Recommendation

In [17]:
def get_congestion_level(density_percent, avg_speed_kmh, vehicle_count, queue_length_veh):
    # Normalization
    density = density_percent / 100
    speed = min(avg_speed_kmh, 80)        # cap speed untuk stabilitas model
    speed_norm = speed / 80

    # Non-linear pressure signals
    speed_pressure = (1 - speed_norm) ** 2.0   # semakin lambat -> eksponensial naik
    queue_pressure = min(queue_length_veh / 8, 1)  # bottleneck indicator
    vehicle_pressure = min(vehicle_count / 70, 1)   # kapasitas jalan

    # Base pressure model
    pressure = (
        0.30 * density +
        0.40 * speed_pressure +
        0.20 * queue_pressure +
        0.10 * vehicle_pressure
    )

    # Hard safety rules
    if speed < 1 and density < 0.02:
        return "Low"   # sensor noise / idle condition

    if speed < 5:
        if queue_length_veh >= 2 or density > 0.2:
            return "Severe"  # breakdown traffic
        return "High"       # near-stall condition

    # Synergy effect (real traffic behavior)
    if speed < 20 and density > 0.4:
        pressure += 0.18   # congestion amplification

    if speed < 15 and queue_length_veh > 0:
        pressure += 0.12   # bottleneck effect

    if density > 0.55 and speed < 40:
        pressure += 0.10   # unstable flow region

    # Anti false positive fix
    if speed > 60 and density > 0.5:
        pressure -= 0.12   # fast but dense correction

    if speed < 10:
        pressure += 0.20   # forced congestion detection

    # Contradiction/ Intelligence layer
    if density > 0.45 and speed > 60:
        pressure += 0.20   # unstable fast-dense flow

    if density > 0.5 and queue_length_veh == 0 and speed < 35:
        pressure += 0.10   # hidden congestion detection

    # Final classification
    if pressure >= 0.58:
        return "Severe"
    elif pressure >= 0.44:
        return "High"
    elif pressure >= 0.28:
        return "Medium"
    else:
        return "Low"

# FLOW INTERPRETATION ENGINE
def interpret_flow(density, speed, queue):
    # dense but fast flow (high throughput highway state)
    if density > 0.6 and speed > 45:
        return "dense_fast_flow"

    # congested traffic state
    elif density > 0.55 and speed < 25:
        return "congested_flow"

    # traffic breakdown (near stop)
    elif speed < 8 and queue > 0:
        return "breakdown_flow"

    # unstable transition flow
    elif 0.35 < density < 0.6 and 25 <= speed <= 55:
        return "unstable_flow"

    # free flow condition
    elif density < 0.25 and speed > 50:
        return "free_flow"

    # default normal state
    return "normal_flow"

def build_ai_insight(vc, spd, density, weather, period, queue_len, congestion_level,
                      accident, roadwork, event, green_seconds):
    parts = []

    density_ratio = density / 100
    flow_type = interpret_flow(density_ratio, spd, queue_len)

    if congestion_level == "Severe":
        parts.append(
            f"Severe congestion detected with {queue_len} queued vehicles. "
            f"Speed dropped to {round(spd,1)} km/h indicating traffic breakdown. "
            f"Green duration extended to {green_seconds:.0f}s to help dissipate the queue."
        )

    elif congestion_level == "High":
        if flow_type == "critical_queue_flow":
            parts.append(
                "High congestion caused by queue spillback and bottleneck failure. "
                f"Green duration adjusted to {green_seconds:.0f}s."
            )
        else:
            parts.append(
                f"High traffic density with {queue_len} queued vehicles and speed {round(spd,1)} km/h. "
                f"Green duration adjusted to {green_seconds:.0f}s."
            )

    elif congestion_level == "Medium":
        if flow_type in ["dense_flow", "unstable_flow"]:
            parts.append(
                "Traffic is dense but still maintaining partial flow stability. "
                f"Green duration set to {green_seconds:.0f}s."
            )
        else:
            parts.append(
                f"Moderate congestion with {vc} vehicles and speed {round(spd,1)} km/h. "
                f"Green duration set to {green_seconds:.0f}s."
            )

    else:
        parts.append(
            f"Traffic is stable with low density ({round(density,1)}%) and speed {round(spd,1)} km/h. "
            f"Keep default green duration ({green_seconds:.0f}s)."
        )

    # context layer
    if period in ("Morning Rush", "Evening Rush"):
        parts.append(f"Peak-hour ({period}) increases traffic demand.")

    if weather == "Rain":
        parts.append("Rain may reduce road friction and visibility.")

    if accident > 0:
        parts.append(f"{accident} accident(s) impacting flow.")

    if roadwork:
        parts.append("Roadwork reduces lane capacity.")

    if event:
        parts.append("Nearby event increases traffic volume.")

    return " ".join(parts)

def build_recommendation(congestion, queue, density, speed, green_seconds):
    actions = []

    if congestion == "Severe":
        actions.append(f"Activate emergency traffic control and extend green phase to {green_seconds:.0f}s.")
        if speed < 5:
            actions.append("Deploy rapid incident response team.")
        if queue > 3:
            actions.append("Clear queue buildup immediately.")

    elif congestion == "High":
        actions.append(f"Increase signal optimization; set green duration to {green_seconds:.0f}s.")
        if queue > 5:
            actions.append("Focus on bottleneck queue reduction.")
        if density > 60:
            actions.append("Adjust adaptive signal timing.")

    elif congestion == "Medium":
        actions.append(f"Monitor closely; adjust green duration to {green_seconds:.0f}s if needed.")
        if speed < 30:
            actions.append("Consider pre-emptive signal adjustment.")

    else:
        actions.append(f"Maintain normal operations with default green duration ({green_seconds:.0f}s).")

    return " ".join(actions)

def get_period_label(hour):
    if  5 <= hour <  9: return 'Morning Rush'
    elif 9 <= hour < 16: return 'Daytime'
    elif 16 <= hour < 19: return 'Evening Rush'
    elif 19 <= hour < 23: return 'Night'
    else:                 return 'Late Night'

# MAIN PIPELINE — gabungkan congestion, insight, recommendation, dan hasil LSTM
prediction_records = []

for idx, row in vc_df.iterrows():
    vc      = int(row['vehicle_count'])
    spd     = float(row['avg_speed_kmh'])
    density = float(row['density_percent'])
    queue   = int(row['queue_length_veh'])
    weather = str(row['weather'])
    green_sec = float(row['green_seconds'])

    # Clean the time string by removing ' AM' or ' PM' if present
    time_str = str(row['time'])
    if ' AM' in time_str:
        time_str = time_str.replace(' AM', '')
    elif ' PM' in time_str:
        time_str = time_str.replace(' PM', '')

    dt_obj = datetime.strptime(f"{row['date']} {time_str}", '%m/%d/%Y %H:%M:%S')
    hour   = dt_obj.hour
    period = row.get('period', get_period_label(hour))

    cong = get_congestion_level(density, spd, vc, queue)

    insight = build_ai_insight(
        vc, spd, density, weather, period, queue, cong,
        int(row.get('accident_count', 0)),
        int(row.get('roadwork_flag', 0)),
        int(row.get('event_flag', 0)),
        green_sec
    )
    recom = build_recommendation(cong, queue, density, spd, green_sec)

    rec = {
        "camera_id":    row.get('camera_id', f'CAM_{idx+1:03d}'),
        "source_video": row.get('source_video', ''),

        # TIME INFO
        "date":   str(row['date']),
        "time":   str(row['time']),
        "period": period,

        # LOCATION INFO
        "latitude":  row.get('latitude'),
        "longitude": row.get('longitude'),

        # WEATHER INFO
        "weather":        weather,
        "weather_temp_c": float(row['weather_temp_c']),

        # TRAFFIC CORE (dari hasil YOLO)
        "vehicle_count":       vc,
        "vehicle_count_1min":  int(row['vehicle_count_1min']),
        "volume_veh_per_hour": int(row['volume_veh_per_hour']),
        "avg_speed_kmh":       round(spd, 2),
        "density_percent":     round(density, 2),
        "queue_length_veh":    queue,
        "green_seconds":       round(green_sec, 1),
        "congestion_level":    cong,

        # MODEL OUTPUT (LSTM)
        "predicted_volume_15m": float(row['predicted_volume_15m']),
        "predicted_volume_2h":  float(row['predicted_volume_2h']),
        "predicted_volume_4h":  float(row['predicted_volume_4h']),

        "model_mape_15m": HORIZON_MAPE['15m'],
        "model_mape_2h":  HORIZON_MAPE['2h'],
        "model_mape_4h":  HORIZON_MAPE['4h'],

        # METADATA
        "ai_insight":     insight,
        "recommendation": recom,
    }
    prediction_records.append(rec)

print(f"Records built: {len(prediction_records)}")
for rec in prediction_records:
    print(
        rec['camera_id'], rec['source_video'],
        "| pred_15m =", rec['predicted_volume_15m'],
        "| pred_2h ="  , rec['predicted_volume_2h'],
        "| pred_4h ="  , rec['predicted_volume_4h'],
        "| green ="    , rec['green_seconds'],
        "| cong ="     , rec['congestion_level']
    )

Records built: 100
CAM_002 pagi_cctv_001_20260518_070935.mp4 | pred_15m = 4506.75 | pred_2h = 1585.76 | pred_4h = 4978.05 | green = 44.2 | cong = Low
CAM_001 pagi_cctv_002_20260518_070954.mp4 | pred_15m = 3870.67 | pred_2h = 1627.97 | pred_4h = 4986.5 | green = 57.8 | cong = Medium
CAM_001 pagi_cctv_003_20260614_080117.mp4 | pred_15m = 1750.73 | pred_2h = 1215.82 | pred_4h = 4722.32 | green = 42.0 | cong = Low
CAM_011 pagi_cctv_004_20260518_071356.mp4 | pred_15m = 3832.92 | pred_2h = 1632.3 | pred_4h = 5003.04 | green = 58.8 | cong = Medium
CAM_007 pagi_cctv_005_20260614_080139.mp4 | pred_15m = 4123.86 | pred_2h = 1507.94 | pred_4h = 4727.08 | green = 46.9 | cong = Medium
CAM_006 pagi_cctv_006_20260518_071450.mp4 | pred_15m = 3183.62 | pred_2h = 1678.69 | pred_4h = 4930.19 | green = 90.0 | cong = Severe
CAM_009 pagi_cctv_007_20260614_080221.mp4 | pred_15m = 4035.83 | pred_2h = 1624.52 | pred_4h = 4751.33 | green = 57.9 | cong = Medium
CAM_011 pagi_cctv_008_20260614_083937.mp4 | pred_15

## 15. Save Prediction Output (CSV & JSON)

In [18]:
json_path = os.path.join(OUTPUT_DIR, 'lstm_prediction_output.json')
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(prediction_records, f, ensure_ascii=False, indent=2)
print(f"Saved: {json_path}")

pred_df  = pd.DataFrame(prediction_records)
pred_csv = os.path.join(OUTPUT_DIR, 'lstm_prediction_output.csv')
pred_df.to_csv(pred_csv, index=False)
print(f"Saved: {pred_csv}")

Saved: /content/outputs/lstm_prediction_output.json
Saved: /content/outputs/lstm_prediction_output.csv


## 16. Prediction Validation & Distribution Analysis (15m / 2h / 4h)

In [19]:
print("=== LSTM PREDICTION VALIDATION (15m / 2h / 4h) ===")

vehicle_counts = np.array([rec['vehicle_count'] for rec in prediction_records])
horizon_stats = {}

for h in ['15m', '2h', '4h']:
    preds  = np.array([rec[f'predicted_volume_{h}'] for rec in prediction_records])
    mape_h = HORIZON_MAPE.get(h)

    pred_min, pred_max   = float(np.min(preds)), float(np.max(preds))
    pred_mean, pred_std  = float(np.mean(preds)), float(np.std(preds))
    pred_spread          = pred_max - pred_min
    relative_spread      = pred_spread / max(pred_mean, 1)

    corr_value = None
    if len(np.unique(vehicle_counts)) > 1:
        corr_value = float(np.corrcoef(vehicle_counts, preds)[0, 1])

    print(f"\n--- Horizon {h} ---")
    print(f"  Model MAPE       : {mape_h}")
    print(f"  Min Prediction   : {pred_min:.2f}")
    print(f"  Max Prediction   : {pred_max:.2f}")
    print(f"  Mean Prediction  : {pred_mean:.2f}")
    print(f"  Std Deviation    : {pred_std:.2f}")
    print(f"  Prediction Spread: {pred_spread:.2f}")
    print(f"  Relative Spread  : {relative_spread:.2%}")

    if relative_spread < 0.10:
        print("  [WARNING] Predictions are highly concentrated.")
    elif relative_spread < 0.20:
        print("  [INFO] Predictions show moderate variability.")
    else:
        print("  [OK] Predictions show strong variability.")

    if corr_value is not None:
        print(f"  Correlation w/ vehicle_count: {corr_value:.3f}")
        if abs(corr_value) < 0.20:
            print("  [WARNING] Weak relationship with vehicle_count.")
        elif abs(corr_value) < 0.50:
            print("  [INFO] Moderate relationship with vehicle_count.")
        else:
            print("  [OK] Strong relationship with vehicle_count.")
    else:
        print("  [INFO] Correlation undefined (no variance in vehicle_count)")

    # Confidence range per record (jika MAPE tersedia)
    if mape_h is not None:
        for rec, p in zip(prediction_records, preds):
            lower = p * (1 - mape_h / 100)
            upper = p * (1 + mape_h / 100)
            rec[f'confidence_range_{h}'] = {
                "lower_bound": float(lower),
                "upper_bound": float(upper),
            }
    else:
        for rec in prediction_records:
            rec[f'confidence_range_{h}'] = None

    horizon_stats[h] = {
        "model_mape": mape_h,
        "min_prediction": pred_min,
        "max_prediction": pred_max,
        "mean_prediction": pred_mean,
        "std_deviation": pred_std,
        "prediction_spread": pred_spread,
        "relative_spread_percent": relative_spread * 100,
        "correlation_vehicle_count_prediction": corr_value,
        "analysis": {
            "prediction_variability": (
                "Low" if relative_spread < 0.10 else
                "Moderate" if relative_spread < 0.20 else
                "High"
            ),
            "output_collapse_detected": bool(relative_spread < 0.02),
            "correlation_strength": (
                "Undefined" if corr_value is None else
                "Weak" if abs(corr_value) < 0.20 else
                "Moderate" if abs(corr_value) < 0.50 else
                "Strong"
            ),
        }
    }

=== LSTM PREDICTION VALIDATION (15m / 2h / 4h) ===

--- Horizon 15m ---
  Model MAPE       : 21.8059
  Min Prediction   : 205.56
  Max Prediction   : 5472.48
  Mean Prediction  : 2752.94
  Std Deviation    : 1795.50
  Prediction Spread: 5266.92
  Relative Spread  : 191.32%
  [OK] Predictions show strong variability.
  Correlation w/ vehicle_count: 0.542
  [OK] Strong relationship with vehicle_count.

--- Horizon 2h ---
  Model MAPE       : 27.035
  Min Prediction   : 162.83
  Max Prediction   : 1955.03
  Mean Prediction  : 1458.56
  Std Deviation    : 478.41
  Prediction Spread: 1792.20
  Relative Spread  : 122.87%
  [OK] Predictions show strong variability.
  Correlation w/ vehicle_count: 0.299
  [INFO] Moderate relationship with vehicle_count.

--- Horizon 4h ---
  Model MAPE       : 32.492
  Min Prediction   : 235.26
  Max Prediction   : 5026.48
  Mean Prediction  : 3970.98
  Std Deviation    : 1353.49
  Prediction Spread: 4791.22
  Relative Spread  : 120.66%
  [OK] Predictions show

## 17. Save Inference Summary JSON

In [20]:
summary = {
    "total_rows_processed": len(prediction_records),

    "seq_len_used":  SEQ_LEN,
    "feature_count": NUM_FEATURES,
    "horizons":      ['15m', '2h', '4h'],
    "horizon_mape":  HORIZON_MAPE,

    "horizon_stats": horizon_stats,

    "predictions": prediction_records,
}

summary_path = os.path.join(OUTPUT_DIR, "lstm_inference_summary.json")
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print(f"Saved: {summary_path}")

Saved: /content/outputs/lstm_inference_summary.json


## 18. Zip Outputs and Download

In [21]:
import shutil
zip_path = '/content/traffix_lstm_inference_outputs'
shutil.make_archive(zip_path, 'zip', OUTPUT_DIR)
print(f"Created: {zip_path}.zip")

from google.colab import files
files.download(f"{zip_path}.zip")

Created: /content/traffix_lstm_inference_outputs.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 19. Final Summary

In [22]:
print("\n=== LSTM INFERENCE COMPLETE ===")
print(f"  Rows processed : {len(prediction_records)}")
print(f"  MAPE 15m       : {HORIZON_MAPE['15m']}")
print(f"  MAPE 2h        : {HORIZON_MAPE['2h']}")
print(f"  MAPE 4h        : {HORIZON_MAPE['4h']}")
print()
for r in prediction_records:
    print(f"  {r['camera_id']} | {r['source_video']}")
    print(f"    {r['date']} {r['time']} | {r['period']}")
    if r['latitude']:
        print(f"    lat={r['latitude']}  lon={r['longitude']}")
    print(f"    weather={r['weather']} {r['weather_temp_c']}C")
    print(f"    vc={r['vehicle_count']} veh | vol={r['volume_veh_per_hour']} veh/hr | congestion={r['congestion_level']}")
    print(f"    green_seconds={r['green_seconds']}s")
    print(f"    predicted_volume -> 15m={r['predicted_volume_15m']} | 2h={r['predicted_volume_2h']} | 4h={r['predicted_volume_4h']}")
    print(f"    {r['ai_insight']}")
    print(f"    -> {r['recommendation']}")
    print()


=== LSTM INFERENCE COMPLETE ===
  Rows processed : 100
  MAPE 15m       : 21.8059
  MAPE 2h        : 27.035
  MAPE 4h        : 32.492

  CAM_002 | pagi_cctv_001_20260518_070935.mp4
    5/18/2026 6:09:33 | Morning Rush
    lat=-6.2131428  lon=106.6837664
    weather=Cloudy 25.6C
    vc=23 veh | vol=4320 veh/hr | congestion=Low
    green_seconds=44.2s
    predicted_volume -> 15m=4506.75 | 2h=1585.76 | 4h=4978.05
    Traffic is stable with low density (19.8%) and speed 53.6 km/h. Keep default green duration (44s). Peak-hour (Morning Rush) increases traffic demand.
    -> Maintain normal operations with default green duration (44s).

  CAM_001 | pagi_cctv_002_20260518_070954.mp4
    5/18/2026 6:15:26 | Morning Rush
    lat=-6.1911513  lon=106.7441887
    weather=Cloudy 26.8C
    vc=67 veh | vol=4260 veh/hr | congestion=Medium
    green_seconds=57.8s
    predicted_volume -> 15m=3870.67 | 2h=1627.97 | 4h=4986.5
    Traffic is dense but still maintaining partial flow stability. Green duratio